# 5.2 · Softmax 回归 / Softmax (Multinomial Logistic) Regression

> **课程定位 / Where this fits**
> 第 2 课，**Part 5 · 监督学习：分类**。
> Lesson 2, **Part 5 · Supervised Classification**.
>
> 5.1 的逻辑回归只会做**二分类**。但现实里常常 ≥3 类（鸢尾花 3 种、手写数字 10 种）。Softmax 回归把 sigmoid **推广到 K 类**，输出一个完整的**概率分布**。它也是神经网络最后一层的标配（Part 12 会再见）。
> Logistic regression (5.1) handles only **two classes**. But real problems often have ≥3 (3 iris species, 10 digits). Softmax regression **generalizes the sigmoid to K classes**, outputting a full **probability distribution**. It's also the standard final layer of neural nets (back in Part 12).

> 📐 **符号约定 / Notation**（详见 [`NOTATION.md`](../NOTATION.md)）
> - $K$ —— 类别数 / number of classes
> - $z_k = \mathbf{x}^\top\mathbf{w}_k$ —— 第 $k$ 类的**分数(logit)** / score (logit) for class $k$
> - $\mathbf{p}$ —— 各类概率向量（和为 1）/ vector of class probabilities (sums to 1)
> - $\mathbf{y}$ —— 真实类别的 one-hot 向量 / one-hot vector of the true class

> 💡 **面试相关 / Interview-relevant**
> - "softmax 公式 + 为什么要减最大值（数值稳定）"（★★★★）
> - "softmax + 交叉熵的梯度为什么是 $(p-y)$"（★★★★）
> - "softmax(K类) vs K 个 one-vs-rest 逻辑回归"（★★★★）
> - "softmax 为什么过参数化 / 为什么常固定一类"（★★★）

---

## 学习目标 / Learning Objectives

1. 从二分类 sigmoid 推广到 K 类 **softmax**。
   Generalize the binary sigmoid to the K-class **softmax**.
2. 写出**数值稳定**的 softmax（减最大值）并解释为什么。
   Write a **numerically stable** softmax (subtract the max) and explain why.
3. 看清交叉熵梯度依然是 $(\mathbf{p}-\mathbf{y})$ 的优雅结构（承接 5.1）。
   See that the cross-entropy gradient is still the clean $(\mathbf{p}-\mathbf{y})$ (continuing 5.1).
4. **从零**实现并与 sklearn 的 multinomial 对照。
   Implement **from scratch** and compare with sklearn's multinomial.
5. 区分 softmax 与 OvR（为 5.13 铺垫）。
   Distinguish softmax from OvR (setting up 5.13).

## 目录 / TOC
1. [先建直觉：让 K 个分数变成概率分布](#1)
2. [从 sigmoid 到 softmax ⭐](#2)
3. [🌸 数据：Iris](#3)
4. [数值稳定的 softmax ⭐](#4)
5. [从零实现 + 对照 sklearn](#5)
6. [softmax vs OvR ⭐](#6)
7. [小结](#7)


<a id="1"></a>
## 1. 先建直觉：让 K 个分数变成概率分布 / Intuition First

二分类时，我们只需要一个概率 $p$（另一类自动是 $1-p$）。K 分类时，我们想要 **K 个概率**，它们必须都 ≥0 且加起来等于 1（一个合法的概率分布）。
For two classes we need just one probability $p$ (the other is $1-p$). For K classes we want **K probabilities**, all ≥0 and summing to 1 (a valid distribution).

做法很自然：每个类配一组权重，先算出 K 个**分数** $z_k$（分数高 = 更像这个类）。但分数是任意实数，怎么变成概率？
The recipe is natural: give each class its own weights and compute K **scores** $z_k$ (higher = more like that class). But scores are arbitrary reals — how to turn them into probabilities?

**softmax 的两步**：(1) 对每个分数取指数 $e^{z_k}$（保证为正、并放大差距）；(2) 除以所有指数之和（归一化，使总和为 1）。这就像把分数变成"投票占比"。
**Softmax in two steps:** (1) exponentiate each score $e^{z_k}$ (makes them positive and amplifies gaps); (2) divide by the sum of all exponentials (normalize so they sum to 1). Like turning scores into "vote shares".

它叫 "soft-max" 是因为它是 **argmax 的平滑版**：最大的分数拿走最大的概率，但不是非 0 即 1，而是平滑分配。
It's called "soft-max" because it's a **smooth version of argmax**: the largest score gets the largest probability, but smoothly rather than all-or-nothing.


<a id="2"></a>
## 2. 从 sigmoid 到 softmax ⭐ / From Sigmoid to Softmax

把上面的直觉写成公式。每类 $k$ 一组权重 $\mathbf{w}_k$，分数 $z_k=\mathbf{x}^\top\mathbf{w}_k$，则：
Formalizing the intuition: each class $k$ has weights $\mathbf{w}_k$, score $z_k=\mathbf{x}^\top\mathbf{w}_k$, and:

$$\Pr(y=k\mid\mathbf{x}) = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}}$$

**损失**还是交叉熵，只是多类版（= 多项分布的 MLE，承接 2.9/5.1）。设真实类的 one-hot 为 $\mathbf{y}$：
The **loss** is again cross-entropy, the multi-class version (= MLE of the multinomial; continuing 2.9/5.1). With the true class as one-hot $\mathbf{y}$:

$$J = -\frac{1}{n}\sum_i \sum_k y_{ik}\log p_{ik}$$

**梯度和 5.1 一模一样的优雅形式**——预测概率减真实标签：
The **gradient has the same clean form as 5.1** — predicted probability minus true label:

$$\nabla_{\mathbf{w}_k} J = \frac{1}{n}\sum_i (p_{ik}-y_{ik})\,\mathbf{x}_i$$

**一个细节：过参数化**。softmax 有冗余——给所有 $\mathbf{w}_k$ 同时加一个常向量，概率不变。所以可以固定某一类的权重为 0（二分类的 sigmoid 正是 K=2 的这种情形）。
**A subtlety: over-parameterization.** Softmax is redundant — adding a constant vector to every $\mathbf{w}_k$ leaves probabilities unchanged. So one class's weights can be fixed to 0 (the binary sigmoid is exactly this K=2 case).


<a id="3"></a>
## 3. 数据：Iris / The Iris Dataset

**Iris（鸢尾花）** 是统计学最经典的数据集（Fisher 1936）：150 朵花，3 个品种各 50 朵，4 个特征（花萼/花瓣的长与宽）。是三分类的标准教学集。
**Iris** is statistics' most classic dataset (Fisher 1936): 150 flowers, 3 species × 50, 4 features (sepal/petal length & width). The standard 3-class teaching set.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
sns.set_theme(style="whitegrid")
np.set_printoptions(precision=4, suppress=True)

iris = load_iris()
X, y = iris.data, iris.target
df = pd.DataFrame(X, columns=iris.feature_names); df["species"] = [iris.target_names[i] for i in y]
print("Iris:", X.shape, "| 3 类各 / per class:", np.bincount(y))
print(df.groupby("species").mean().round(2))

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
sc = StandardScaler().fit(X_tr)
Xtr, Xte = sc.transform(X_tr), sc.transform(X_te)


<a id="4"></a>
## 4. 数值稳定的 softmax ⭐ / Numerically Stable Softmax

朴素地算 $e^{z_k}$，当 $z$ 较大时会**溢出**成 `inf`（比如 $e^{1002}$）。技巧：分子分母同乘 $e^{-\max_j z_j}$——数学上结果完全不变，但现在所有指数的参数都 ≤0，绝不溢出。**面试常考这一步。**
Naively computing $e^{z_k}$ **overflows** to `inf` for large $z$ (e.g. $e^{1002}$). The trick: multiply top and bottom by $e^{-\max_j z_j}$ — mathematically identical, but now every exponent is ≤0 and never overflows. **This step is a common interview question.**


In [ ]:
def softmax(Z):
    # Z 形状是 (样本数, 类别数K); 每一行是一个样本对 K 个类的分数
    # Z.max(axis=1) 取每行最大值; keepdims=True 保持二维(n,1)以便逐行相减(广播)
    # 减最大值后, 最大分数变成0, 其余为负 → exp 不会溢出, 而比值(概率)不变
    Z = Z - Z.max(axis=1, keepdims=True)
    e = np.exp(Z)                          # 逐元素取指数, 都在 (0,1] 之间, 不会 inf
    return e / e.sum(axis=1, keepdims=True)  # 每行除以该行之和 → 每行概率加起来=1

Z_demo = np.array([[1000., 1001., 1002.]])   # 三个很大的分数; 朴素 exp(1002) 会溢出成 inf
print("稳定 softmax / stable softmax:", softmax(Z_demo), " 和 sum =", softmax(Z_demo).sum())
print("朴素 naive exp(1002) =", np.exp(1002.), "→ inf, 故必须减最大值 / must subtract the max")


<a id="5"></a>
## 5. 从零实现 + 对照 sklearn / From Scratch & vs sklearn

代码就是第 2 节公式的直译：用稳定 softmax 算概率，梯度用 $\mathbf{X}^\top(\mathbf{P}-\mathbf{Y})$。
The code directly translates Section 2: probabilities via the stable softmax, gradient $\mathbf{X}^\top(\mathbf{P}-\mathbf{Y})$.


In [ ]:
def one_hot(y, K):
    # 把整数标签(如 2)变成 one-hot 行向量(如 [0,0,1]); 交叉熵需要这种形式
    Y = np.zeros((len(y), K))              # 先全 0, 形状 (样本数, K)
    Y[np.arange(len(y)), y] = 1            # 第 i 行、第 y[i] 列 置 1
    return Y

def fit_softmax(X, y, K, lr=0.5, n_iter=2000):
    Xb = np.c_[np.ones(len(X)), X]         # np.c_ 在最左边拼一列全 1 作偏置项
    Y = one_hot(y, K)                      # 真实标签转 one-hot, 形状 (n, K)
    W = np.zeros((Xb.shape[1], K))         # 权重矩阵: 每列是一个类的权重向量
    for _ in range(n_iter):                # 梯度下降迭代
        P = softmax(Xb @ W)                # Xb@W 得到 (n,K) 分数, softmax 转成概率
        grad = Xb.T @ (P - Y) / len(y)     # 梯度 = Xᵀ(P-Y)/n, 与 5.1 同结构 (P-Y 是预测-真实)
        W -= lr * grad                     # 沿负梯度更新权重
    return W

W = fit_softmax(Xtr, y_tr, K=3)
Pte = softmax(np.c_[np.ones(len(Xte)), Xte] @ W)  # 测试集同样加偏置列再算概率
acc = (Pte.argmax(1) == y_te).mean()       # argmax(1) 取每行概率最大的类当预测
print(f"从零 softmax test 准确率 from-scratch accuracy: {acc:.3f}")
print("预测概率示例(每行和=1) example probs (row sums to 1):", Pte[0].round(3), "→ 预测类 pred class", Pte[0].argmax())


In [ ]:
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(max_iter=2000).fit(Xtr, y_tr)   # sklearn 默认用 multinomial(真 softmax)
print(f"sklearn softmax test 准确率 accuracy: {clf.score(Xte, y_te):.3f}")
print(f"从零模型与 sklearn 预测一致率 agreement: {(Pte.argmax(1) == clf.predict(Xte)).mean():.3f}")

# 2D 决策边界 (用花瓣长宽) / 2D decision boundary on petal length & width
X2tr = Xtr[:, 2:4]
clf2 = LogisticRegression(max_iter=2000).fit(X2tr, y_tr)
xx, yy = np.meshgrid(np.linspace(X2tr[:,0].min()-.5, X2tr[:,0].max()+.5, 300),
                     np.linspace(X2tr[:,1].min()-.5, X2tr[:,1].max()+.5, 300))
Z = clf2.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
fig, ax = plt.subplots(figsize=(7, 5))
ax.contourf(xx, yy, Z, alpha=0.3, cmap="viridis")
ax.scatter(X2tr[:,0], X2tr[:,1], c=y_tr, cmap="viridis", edgecolor="k", s=30)
ax.set_xlabel("petal length (std)"); ax.set_ylabel("petal width (std)")
ax.set_title("Softmax 多类决策边界 / multi-class boundary\n每类一个线性判别, 两两分界是直线 → 分段线性")
plt.tight_layout(); plt.show()


<a id="6"></a>
## 6. softmax vs OvR ⭐ / Softmax vs One-vs-Rest

做多分类有两条思路（5.13 会展开）：
There are two routes to multi-class (expanded in 5.13):

| | Softmax (multinomial) | One-vs-Rest (OvR) |
|---|---|---|
| 模型 model | **一个**模型，K 组权重联合训练 / one model, K weight sets trained jointly | **K 个**独立二分类器 / K independent binary classifiers |
| 概率 prob. | 天然归一化（和=1）/ naturally normalized | 各自 sigmoid，需再归一化 / each sigmoid, needs renormalizing |
| 直觉 intuition | 各类**竞争**同一份概率 / classes compete for one probability | 每类**独立**判断"是不是我" / each asks "is it me?" |

互斥单标签问题（如 Iris，一朵花只属一个品种）优先用 softmax；多标签问题（一个样本可属多个类）用 OvR。
For mutually-exclusive single-label problems (Iris: a flower is one species) prefer softmax; for multi-label (a sample can have several labels) use OvR.


In [ ]:
from sklearn.multiclass import OneVsRestClassifier
ovr = OneVsRestClassifier(LogisticRegression(max_iter=2000)).fit(Xtr, y_tr)
print(f"Softmax 准确率 accuracy: {clf.score(Xte, y_te):.3f}")
print(f"OvR     准确率 accuracy: {ovr.score(Xte, y_te):.3f}  (略低: 3 个独立边界不如联合训练协调)")
print("softmax 概率天然和=1; OvR 各自 sigmoid 后由 sklearn 归一化才使和=1:")
print("OvR 概率示例 example:", ovr.predict_proba(Xte[:1]).round(3), "和 sum=", ovr.predict_proba(Xte[:1]).sum().round(3))


<a id="7"></a>
## 7. 小结 / Summary

```
softmax: p_k = e^{z_k}/Σ e^{z_j} → 概率分布(>0, 和=1), 是 argmax 的平滑版
数值稳定: 减每行最大值(结果不变, 防溢出) — 面试常考
损失=多类交叉熵(=多项 MLE); 梯度=Xᵀ(P-Y), 与 5.1 同结构
过参数化: 可固定一类权重=0; 二分类 sigmoid 是 K=2 特例
softmax(互斥单标签) vs OvR(独立, 可多标签) → 5.13 展开
```

### 💡 面试速查 / Interview cheat-sheet
1. **softmax = sigmoid 的 K 类推广**，输出归一化概率分布。
   Softmax = K-class generalization of sigmoid; outputs a normalized distribution.
2. **减最大值**防溢出，结果不变。
   Subtract the max to prevent overflow; result unchanged.
3. 梯度还是干净的 **$(\mathbf{p}-\mathbf{y})$**。
   The gradient is still the clean $(\mathbf{p}-\mathbf{y})$.
4. 互斥多类用 softmax；多标签用 OvR。
   Use softmax for mutually-exclusive classes; OvR for multi-label.

### 下一节 / Next
**5.3 KNN**——前面都是参数模型（学权重）。KNN 是**非参数惰性**模型：不训练，预测时找最近邻投票。
**5.3 KNN** — these were parametric models (learn weights). KNN is **non-parametric and lazy**: no training, just nearest-neighbor voting at predict time.
